# 🇮🇳 VoiceGuard India - Ultimate Multi-Dataset Streaming Pipeline

This pipeline combines the top 9 Hindi Speech datasets (Authentic + Deepfake) from HuggingFace into a single unified training architecture.

To prevent Disk and RAM crashes, this notebook uses **HuggingFace Streaming**. It streams the audio over the internet, extracts the MFCCs, and immediately deletes the audio file from memory.

In [ ]:
!pip install datasets librosa scikit-learn numpy tqdm xgboost soundfile

In [ ]:
import os
import librosa
import numpy as np
import pickle
from datasets import load_dataset, interleave_datasets
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

### ⬇️ Step 1: Initialize Dataset Streams
We use `streaming=True` so we don't download Terabytes of data to the hard drive!

In [ ]:
print("🌍 Initializing HuggingFace Streams...")

# -- AUTHENTIC HUMAN HINDI STREAMS (Label 1) --
auth_1 = load_dataset("mozilla-foundation/common_voice_11_0", "hi", split="train", streaming=True)
auth_2 = load_dataset("google/fleurs", "hi_in", split="train", streaming=True)
auth_3 = load_dataset("SPRINGLab/IndicTTS-Hindi", split="train", streaming=True)
# ud-nlp/hindi-speech-recognition is extremely large, streaming is critical here
auth_4 = load_dataset("ud-nlp/hindi-speech-recognition-dataset", split="train", streaming=True)

# -- DEEPFAKE HINDI STREAMS (Label 0) --
# IndicSynth contains 4000 hours of fakes. We stream it.
fake_1 = load_dataset("IndicSynth", split="train", streaming=True) # Requires access approval if gated
fake_2 = load_dataset("abhirl/hindi-tts-dataset", split="train", streaming=True)
fake_3 = load_dataset("SynHate", split="train", streaming=True) # Assuming Hindi subset is available via filtering

print("✅ All streams successfully connected!")

### 🔊 Step 2: Stream & Extract Features (The Heavy Lifting)
We pull audio from the stream, extract the MFCCs, and discard the audio.

In [ ]:
def extract_features_from_array(audio_array, sr):
    try:
        mfccs = np.mean(librosa.feature.mfcc(y=audio_array, sr=sr, n_mfcc=20).T, axis=0)
        centroid = np.mean(librosa.feature.spectral_centroid(y=audio_array, sr=sr))
        zcr = np.mean(librosa.feature.zero_crossing_rate(audio_array))
        return np.hstack((mfccs, centroid, zcr))
    except:
        return None

X = []
y = []

SAMPLES_PER_DATASET = 5000  # Adjust this depending on how much time you have!

def consume_stream(dataset, label, name, max_samples):
    print(f"\n⏬ Streaming and Extracting: {name}")
    count = 0
    # Wrap in try-except in case a specific HF dataset is gated or offline
    try:
        for item in tqdm(dataset, total=max_samples):
            if count >= max_samples: 
                break
            
            # Datasets format audio arrays differently, attempt standard extraction
            if 'audio' in item and 'array' in item['audio']:
                audio_array = item['audio']['array']
                sr = item['audio']['sampling_rate']
                
                features = extract_features_from_array(audio_array, sr)
                if features is not None:
                    X.append(features)
                    y.append(label)
                    count += 1
    except Exception as e:
        print(f"⚠️ Could not stream {name}. Skipping. Error: {e}")

# Consume Authentic Streams
consume_stream(auth_1, 1, "Mozilla Common Voice (Hindi)", SAMPLES_PER_DATASET)
consume_stream(auth_2, 1, "Google Fleurs (Hindi)", SAMPLES_PER_DATASET)
consume_stream(auth_3, 1, "IndicTTS-Hindi", SAMPLES_PER_DATASET)
consume_stream(auth_4, 1, "UD-NLP Hindi Telephone", SAMPLES_PER_DATASET)

# Consume Fake Streams
consume_stream(fake_1, 0, "IndicSynth Deepfakes", SAMPLES_PER_DATASET)
consume_stream(fake_2, 0, "Hindi TTS Wikipedia Clones", SAMPLES_PER_DATASET)
consume_stream(fake_3, 0, "SynHate (Synthetic Audio)", SAMPLES_PER_DATASET)

X_final = np.array(X)
y_final = np.array(y)
print(f"\n✅ Successfully extracted features for {len(X_final)} total audio files!")

### 🧠 Step 3: Train XGBoost
Because the dataset is incredibly diverse and large, we upgrade from Random Forest to **XGBoost** for maximum performance.

In [ ]:
print("\n🚀 Training XGBoost Classifier on massive combined dataset...")
clf = xgb.XGBClassifier(n_estimators=300, max_depth=10, learning_rate=0.05, n_jobs=-1, random_state=42)
clf.fit(X_final, y_final)

print("\n✅ Training Complete! Accuracy Metrics:")
predictions = clf.predict(X_final)
print(classification_report(y_final, predictions, target_names=["Deepfake", "Authentic"]))

with open("voiceguard_ultimate_model.pkl", "wb") as f:
    pickle.dump(clf, f)
    
print("\n💾 SUCCESS! Ultimate Model saved as 'voiceguard_ultimate_model.pkl'")